# Example 4: QB CPOE vs EPA Quadrant Plot

In [34]:
# Imports
import nfl_data_py as nfl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import numpy as np

In [35]:
# Get play-by-play dataframe
pbp = nfl.import_pbp_data([2025])
pbp.head()

2025 done.
Downcasting floats.


,play_id,game_id,old_game_id_x,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,was_pressure,route,defense_man_zone_type,defense_coverage_type,offense_names,defense_names,offense_positions,defense_positions,offense_numbers,defense_numbers
0,1.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,40.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0.0,,,None,Denzel Burke;Joey Blount;Dadrion Taylor-Demers...,Rejzohn Wright;Quincy Riley;Ugo Amadi;Julian B...,CB;FS;FS;ILB;ILB;RB;S;TE;TE;TE;WR,CB;CB;FS;FS;ILB;ILB;K;MLB;OLB;RB;SS,29;32;42;44;50;31;36;84;81;87;4,25;29;0;23;53;44;19;28;40;13;33
2,63.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0.0,,,None,Hjalte Froholdt;Evan Brown;Isaiah Adams;Kyler ...,Isaac Yiadom;Kool-Aid McKinstry;Nathan Shepher...,C;G;G;QB;RB;T;T;TE;TE;TE;WR,CB;CB;DT;DT;FS;ILB;MLB;NT;OLB;OLB;SS,72;63;74;1;6;73;70;85;84;87;18,27;4;93;90;23;20;56;92;94;96;21
3,85.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0.0,QUICK OUT,ZONE_COVERAGE,COVER_2,Hjalte Froholdt;Evan Brown;Isaiah Adams;Kyler ...,Isaac Yiadom;Kool-Aid McKinstry;Nathan Shepher...,C;G;G;QB;RB;T;T;TE;TE;WR;WR,CB;CB;DT;DT;FS;ILB;MLB;NT;OLB;OLB;SS,72;63;74;1;6;73;70;85;87;14;18,27;4;93;90;23;20;56;92;94;96;21
4,115.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,1.0,,ZONE_COVERAGE,COVER_6,Hjalte Froholdt;Evan Brown;Isaiah Adams;Kyler ...,Isaac Yiadom;Kool-Aid McKinstry;Nathan Shepher...,C;G;G;QB;RB;T;T;TE;TE;WR;WR,CB;CB;DT;DT;FS;ILB;MLB;NT;OLB;OLB;SS,72;63;74;1;6;73;70;85;87;17;18,27;4;93;90;23;20;56;92;94;96;21


In [36]:
# Filter to qb dropbacks
# Removes runs, special teams, non-qb plays
pbp_dropbacks = pbp[
    pbp['qb_dropback'] == 1
].copy()

In [37]:
pbp_dropbacks = pbp_dropbacks[[
    'passer_player_name',
    'epa',
    'cpoe'
]]

pbp_dropbacks.head()

,passer_player_name,epa,cpoe
3,K.Murray,1.317340,19.465660
4,K.Murray,-1.694360,NaN
9,S.Rattler,-0.788527,-83.417946
10,S.Rattler,-1.545796,NaN
12,K.Murray,0.041066,10.352385


In [38]:
pbp_dropbacks.shape

(20884, 3)

In [39]:
# filter out rows with any na values
pbp_dropbacks = pbp_dropbacks.dropna(
    subset=['passer_player_name', 'epa', 'cpoe']
)
pbp_dropbacks.shape

(17489, 3)

In [40]:
# Aggregate to Qb Level
qb_stats = (
    pbp_dropbacks
    .groupby('passer_player_name')
    .agg(
        avg_epa=('epa', 'mean'),
        avg_cpoe=('cpoe', 'mean'),
        plays=('epa', 'size')
    )
    .reset_index()
)

qb_stats.shape

(95, 4)

In [41]:
# Apply minimum play treshold
qb_stats = qb_stats[
    qb_stats['plays'] > 200
]
qb_stats.shape

(34, 4)

In [42]:
# Calculate quadrant lines
mean_epa = qb_stats['avg_epa'].mean()
mean_cpoe = qb_stats['avg_cpoe'].mean()

print(f"mean epa: {mean_epa}")
print(f"mean cpoe: {mean_cpoe}")

mean epa: 0.2063330113887787
mean cpoe: 0.9007883667945862
